# OnT quickstart: ontology embeddings in four steps

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HuiYang1997/OnT/blob/main/notebooks/OnT_Quickstart.ipynb)

**OnT** (Ontology Transformer, ISWC 2025) fine-tunes a sentence encoder so that concepts become points in a hyperbolic (Poincaré) ball:
general concepts sit near the centre, specific ones near the boundary, and $C \sqsubseteq D$ becomes "$C$ close to $D$, and further out".

1. **Install** the `ontology-transformer` package
2. **One click**: load a small pizza ontology and train OnT on it
3. **Embed** concepts, including $\exists r.C$
4. **Score** subsumptions $C \sqsubseteq D$ and rank candidate parents

Runs on the free Colab **CPU** runtime (a GPU is optional: *Runtime → Change runtime type*). Choose **Runtime → Run all**.

## Step 1 · Install

`ontology-transformer` reads OWL files through the OWL API, so it needs Java.

In [1]:
import os, sys, subprocess

if "google.colab" in sys.modules:                       # Java is not preinstalled on Colab
    subprocess.run("apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless > /dev/null",
                   shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

%pip install -q ontology-transformer==0.1.8

import logging, warnings           # keep the output short; delete these two lines to see all training logs
warnings.filterwarnings("ignore"); logging.disable(logging.WARNING)

Note: you may need to restart the kernel to use updated packages.


## Step 2 · One click: load an ontology and train OnT

`pizza.ofn` holds the tutorial's running example (axioms $\alpha_1$–$\alpha_6$) plus a few more pizzas and toppings:
29 classes and 46 axioms in OWL functional syntax. Every class has a readable label, because OnT encodes **labels**.

In [2]:
import urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/HuiYang1997/OnT/main/notebooks/pizza.ofn", "pizza.ofn")

for line in open("pizza.ofn"):                     # the axioms about Margherita and CheesyPizza
    if line.startswith(("SubClassOf(:Margherita", "EquivalentClasses(:CheesyPizza")):
        print(line, end="")

SubClassOf(:Margherita :Pizza)
SubClassOf(:Margherita ObjectSomeValuesFrom(:hasTopping :Mozzarella))
SubClassOf(:Margherita ObjectSomeValuesFrom(:hasTopping :Tomato))
EquivalentClasses(:CheesyPizza ObjectIntersectionOf(:Pizza ObjectSomeValuesFrom(:hasTopping :Cheese)))


Now the one line that matters: `fit` converts the OWL axioms into training data, fine-tunes the encoder
(default base model: `all-MiniLM-L12-v2`, 384 dimensions) and picks the best centripetal weight $\lambda$ for scoring.

In [3]:
from ont import OntologyTransformer

model = OntologyTransformer.fit(owl_path="pizza.ofn", output_dir="ont_pizza",
                                num_epochs=40, batch_size=8, learning_rate=5e-5)

print("best lambda:", model.best_lambda)

Download `en_core_web_sm` for pos tagger.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/12.8 MB ? eta -:--:--

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 39.8 MB/s  0:00:00


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Epoch,Training Loss,Validation Loss,Axiom Kind,Centri Weight,H@1,H@10,H@100,Mrr,Mr,Median,Auc
1,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.416667,2.500000,2.500000,0.955357
2,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.416667,2.500000,2.500000,0.955357
3,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.416667,2.500000,2.500000,0.955357
4,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.375000,3.000000,3.000000,0.946429
5,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.375000,3.000000,3.000000,0.946429
6,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.375000,3.000000,3.000000,0.946429
7,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.500000,2.000000,2.000000,0.982143
8,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.500000,2.000000,2.000000,0.982143
9,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.500000,2.000000,2.000000,0.982143
10,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.500000,2.000000,2.000000,0.982143


best lambda: 0.0


## Step 3 · Embed concepts

`encode` maps any label to a point in the Poincaré ball. Its hyperbolic distance from the origin acts as a **depth**:
general concepts should be closer to the centre than specific ones.
`encode_existence(r, C)` embeds the complex concept $\exists r.C$ by a learned rotation of $C$.

In [4]:
import pandas as pd
import torch

concepts = ["food", "pizza", "margherita", "dairy product", "cheese", "mozzarella"]
emb = model.encode(concepts, convert_to_tensor=True)
print("embedding matrix:", tuple(emb.shape))

depth = model.manifold.dist0(emb)              # hyperbolic distance to the origin
pd.DataFrame({"concept": concepts,
              "depth": depth.cpu().numpy().round(2),
              "first 4 coordinates": ["  ".join(f"{x:+.3f}" for x in v[:4]) for v in emb.cpu().numpy()]})

embedding matrix: (6, 384)


,concept,depth,first 4 coordinates
0,food,5.73,-0.085 +0.228 +0.083 +0.001
1,pizza,8.15,-0.476 +0.264 -0.034 +0.014
2,margherita,13.69,-0.343 +0.562 -0.101 +0.287
3,dairy product,9.49,-0.202 +0.082 +0.354 -0.066
4,cheese,8.91,-0.556 +0.117 -0.068 -0.125
5,mozzarella,9.62,-0.569 +0.111 -0.300 -0.082


In [5]:
some_cheese = model.encode_existence("has topping", "cheese")      # ∃hasTopping.Cheese
print("∃ has topping . cheese ->", some_cheese.shape, "  ".join(f"{x:+.3f}" for x in some_cheese[0, :4]), "...")

∃ has topping . cheese -> (1, 384) -0.607  +0.122  -0.099  -0.097 ...


## Step 4 · Score subsumptions

`score_hierarchy(C, D, λ)` returns $-\big(d(C,D) + \lambda\,(\lVert D\rVert - \lVert C\rVert)\big)$: **higher means $C \sqsubseteq D$ is more plausible**.
Compare candidate parents **of the same concept**: some are stated, some only *entailed* (never seen as such in training), some false.
On a toy ontology, stated parents win clearly, while distant entailed ancestors may score below a false parent: scores are plausibility, not proofs.

In [6]:
lam = model.best_lambda or 0.0

def score(child, parent):
    c = model.encode([child], convert_to_tensor=True)
    p = model.encode([parent], convert_to_tensor=True)
    return round(model.score_hierarchy(c, p, weight=lam).item(), 2)

rows = [("mozzarella", "cheese",        "stated (α4)"),
        ("mozzarella", "dairy product", "entailed (α4, α5)"),
        ("mozzarella", "pizza",         "false"),
        ("margherita", "pizza",         "stated (α1)"),
        ("margherita", "food",          "entailed"),
        ("margherita", "cheese",        "false")]
pd.DataFrame([(f"{c} ⊑ {p}", status, score(c, p)) for c, p, status in rows],
             columns=["axiom", "status", "score"])

,axiom,status,score
0,mozzarella ⊑ cheese,stated (α4),-5.38
1,mozzarella ⊑ dairy product,"entailed (α4, α5)",-11.70
2,mozzarella ⊑ pizza,false,-6.79
3,margherita ⊑ pizza,stated (α1),-10.18
4,margherita ⊑ food,entailed,-12.43
5,margherita ⊑ cheese,false,-11.87


The same score ranks **all** classes as candidate parents, for an existing class or for a concept that is not in the ontology yet:

In [7]:
import json
all_labels = list(json.load(open("ont_pizza/data/concept_names.json")).values())   # class labels, written by fit()
cand = model.encode(all_labels, convert_to_tensor=True)

def top_parents(query, k=4):
    q = model.encode([query], convert_to_tensor=True)
    s = model.score_hierarchy(q.expand_as(cand), cand, weight=lam)
    return [all_labels[i] for i in s.argsort(descending=True).tolist() if all_labels[i] != query][:k]

for query in ["mozzarella", "margherita", "pepperoni pizza"]:      # "pepperoni pizza" is not in the ontology
    print(f"{query:>16} ⊑ ?  ", top_parents(query))

      mozzarella ⊑ ?   ['cheese', 'pizza', 'vegetariana', 'food']
      margherita ⊑ ?   ['pizza', 'mozzarella', 'marinara', 'tomato']
 pepperoni pizza ⊑ ?   ['pizza', 'bell pepper', 'cheese', 'meaty pizza']


### Notes

* Scores are **plausibility**, not proofs: check any new axiom with an OWL reasoner (e.g. ELK, HermiT).
* **Your own ontology**: replace `pizza.ofn` by any OWL file (`.owl`, `.ofn`, `.ttl`), e.g. uploaded via the Colab file panel.
* **No training**: load a released model, e.g. `OntologyTransformer.from_pretrained("Hui97/OnT-MiniLM-L12-galen")` (also `-go`, `-anatomy`, and `OnT-MPNet-*`).
* Paper: Yang, Chen, He, Gao & Horrocks. *Language Models as Ontology Encoders*. ISWC 2025. Code: github.com/HuiYang1997/OnT